In [0]:
from pyspark.sql import functions as F

# ---------- Settings ----------
catalog = "dlt_dbx"
schema  = "default"
base    = f"/Volumes/{catalog}/{schema}/healthcare_data"


# ---------- Ingest function ----------
def ingest(folder, table, file_pattern="*.csv"):
    """Load new CSV files from a folder into a Delta table."""

    # 1. Read new files only (Auto Loader remembers what it already loaded)
    df = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("pathGlobFilter", file_pattern)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", f"{base}/_schemas/{folder}")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("rescuedDataColumn", "_rescued_data")
        .load(f"{base}/{folder}/")
    )

    # 2. Add lineage columns (which file, and when it was loaded)
    df = (
        df
        .withColumn("source_file", F.col("_metadata.file_path"))
        .withColumn("ingested_at", F.current_timestamp())
    )

    # 3. Append to the Delta table, then stop (no always-on stream)
    query = (
        df.writeStream
        .option("checkpointLocation", f"{base}/_checkpoints/{folder}")
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(f"{catalog}.{schema}.{table}")
    )
    query.awaitTermination()


# ---------- Run ----------
ingest("diagnosis_mapping", "raw_diagnosis_map")
ingest("patients_daily",    "raw_patients_daily", file_pattern="patients_daily_file_*.csv")